# Fase 4 - Fontes, variaveis, intervalo e serie historica

Este notebook mostra, no VS Code, a tabela simples solicitada:
fonte, variavel, intervalo e serie historica.

Ele usa o inventario completo das variaveis processadas em Zarr,
sem a tabela curta/curada usada antes.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from inventory_all_processed_variables import OUT_DIR, build_inventory

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 120)

ROOT


In [ ]:
detalhe, tabela = build_inventory()

OUT_DIR.mkdir(parents=True, exist_ok=True)
csv_detalhe = OUT_DIR / "phase4_all_processed_variables_detail.csv"
csv_tabela = OUT_DIR / "phase4_all_processed_variables.csv"
detalhe.to_csv(csv_detalhe, index=False)
tabela.to_csv(csv_tabela, index=False)

tabela_visual = tabela.rename(
    columns={
        "variavel": "vari\u00e1vel",
        "serie_historica": "s\u00e9rie hist\u00f3rica",
    }
)
tabela_visual["intervalo"] = tabela_visual["intervalo"].replace({"diario": "di\u00e1rio"})
tabela_visual = (
    tabela_visual[["fonte", "vari\u00e1vel", "intervalo", "s\u00e9rie hist\u00f3rica"]]
    .sort_values(["fonte", "vari\u00e1vel", "intervalo"])
    .reset_index(drop=True)
)

tabela_visual


In [ ]:
resumo = (
    tabela_visual
    .groupby(["fonte", "intervalo", "s\u00e9rie hist\u00f3rica"], as_index=False)
    .agg(n_variaveis=("vari\u00e1vel", "count"))
    .sort_values(["fonte", "intervalo", "s\u00e9rie hist\u00f3rica"])
    .reset_index(drop=True)
)

resumo


In [ ]:
checagem = pd.DataFrame(
    {
        "metrica": [
            "fontes",
            "linhas fonte-variavel-intervalo",
            "linhas de detalhe por produto Zarr",
        ],
        "valor": [
            tabela_visual["fonte"].nunique(),
            len(tabela_visual),
            len(detalhe),
        ],
    }
)

checagem


## Como ler

- `fonte`: origem do dado ou produto derivado.
- `variavel`: variavel disponivel para entrar na Fase 4A.
- `intervalo`: frequencia temporal da serie usada aqui.
- `serie_historica`: periodo com dados disponiveis no projeto.

As tabelas salvas em disco ficam em:

- `data/processed/parquet/statistics/phase4_all_processed_variables.csv`
- `data/processed/parquet/statistics/phase4_all_processed_variables_detail.csv`
